# Ejercicio 4 — Análisis de Series Temporales
## Practica Final: Estadística para Data Science

**Autor:** Alejandro Pujana Quintero  
**Proposito:** Cuaderno de trabajo paralelo a `ejercicio4_series_temporales.py`.  
Documenta la logica, implementacion de los TODOs e interpretacion estadistica.

---

### Contexto del ejercicio

A diferencia de los ejercicios 1 y 2, aqui trabajamos con **datos sinteticos** generados
por la funcion `generar_serie_temporal(semilla=42)` del profesor.  
No usamos el dataset Loan Approval — la serie es completamente artificial y sus
componentes son **conocidos de antemano**:

| Componente | Formula | Parametros conocidos |
|---|---|---|
| Tendencia | `0.05 * t + 50` | Pendiente: +0.05/dia, intercepto: 50 |
| Estacionalidad | `15*sin(...) + 6*cos(...)` | Amplitud ~15, periodo: 365.25 dias |
| Ciclo | `8*sin(2π*t/1461)` | Amplitud: 8, periodo: ~4 anos (1461 dias) |
| Ruido | `N(0, 3.5)` | Media: 0, sigma: 3.5 |

Esto nos permite **validar** si la descomposicion recupera correctamente los componentes teoricos.

### Regla critica
> **NO modificar `generar_serie_temporal()` ni la semilla.**  
> Penalizacion: -1.0 pt si se modifica. La funcion y el bloque `__main__` se dejaron intactos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import jarque_bera, norm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style='whitegrid', palette='muted')

# Ruta al script del ejercicio (para importar generar_serie_temporal)
import sys
sys.path.insert(0, str(Path('..') / 'practica_final_Pujana_Quintero_Alejandro'))
from ejercicio4_series_temporales import generar_serie_temporal

# Generar la serie — misma semilla que en el script de entrega
serie = generar_serie_temporal(semilla=42)
print(f'Serie generada: {len(serie)} observaciones')
print(f'Periodo: {serie.index[0].date()} -> {serie.index[-1].date()}')
print(f'Media: {serie.mean():.2f} | Std: {serie.std():.2f}')
print(f'Min: {serie.min():.2f} | Max: {serie.max():.2f}')

## Conceptos clave antes de analizar

### Modelo aditivo vs multiplicativo

Una serie temporal puede descomponerse de dos formas:

**Aditivo**: `Y(t) = Tendencia + Estacionalidad + Residuo`  
Usar cuando la amplitud de la estacionalidad es **constante** a lo largo del tiempo.

**Multiplicativo**: `Y(t) = Tendencia × Estacionalidad × Residuo`  
Usar cuando la amplitud de la estacionalidad **crece proporcionalmente** con el nivel de la serie.

En esta serie usamos **modelo aditivo** porque la serie fue construida sumando componentes
y la amplitud estacional no crece con la tendencia — se ve en los graficos que los picos
y valles tienen la misma magnitud en 2018 que en 2023.

### Que es la ACF y la PACF

**ACF (Autocorrelation Function)**: mide la correlacion de la serie con versiones
retrasadas de si misma. ACF en lag=k = correlacion entre el valor de hoy y el valor
de hace k dias. Si todas las barras estan dentro del intervalo de confianza (banda azul),
no hay autocorrelacion significativa — el residuo es ruido blanco.

**PACF (Partial ACF)**: mide la correlacion con lag=k eliminando el efecto de los
lags intermedios. Util para identificar el orden de modelos ARIMA.

### Test ADF (Augmented Dickey-Fuller)

Prueba si una serie tiene raiz unitaria (es no estacionaria).  
- H0: existe raiz unitaria (serie no estacionaria)  
- H1: no existe raiz unitaria (serie estacionaria)  
- Si p < 0.05: rechaza H0 -> la serie ES estacionaria

Un residuo de ruido blanco debe ser estacionario — su media y varianza no cambian con el tiempo.

### Test Jarque-Bera

Prueba si los datos siguen una distribucion normal basandose en skewness y curtosis.  
- H0: los datos son normales (skewness=0, curtosis=0)  
- Si p > 0.05: no rechaza H0 -> los datos son compatibles con normalidad

Elegimos Jarque-Bera sobre Shapiro-Wilk porque Shapiro-Wilk esta limitado a n < 5,000.
Aunque el residuo limpio tiene 1,827 observaciones (dentro del limite), JB es el
estandar en econometria de series temporales y no tiene limite de muestra.

## TODO 1 — Visualizar la serie completa

### Que hace esta funcion

Genera un grafico de linea de los 2,191 dias de la serie (2018-2024) con:
- La serie observada en azul
- Una media movil de 365 dias en rojo discontinuo para visualizar la tendencia subyacente
- Lineas verticales grises por ano para facilitar la lectura temporal

### Por que la media movil de 365 dias

La media movil de 365 dias suaviza la estacionalidad anual — al promediar exactamente
un ano completo, los picos y valles estacionales se cancelan entre si y lo que queda
es la tendencia + ciclo. Es una tecnica clasica en analisis de series temporales
para aislar visualmente la componente de largo plazo sin necesidad de descomposicion formal.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(serie.index, serie.values,
        color='#1976D2', linewidth=0.8, alpha=0.9, label='Serie observada')

ma_365 = serie.rolling(window=365, center=True).mean()
ax.plot(ma_365.index, ma_365.values,
        color='crimson', linewidth=1.8, linestyle='--',
        alpha=0.85, label='Media movil 365 dias (tendencia)')

for anio in range(2018, 2024):
    ax.axvline(pd.Timestamp(f'{anio}-01-01'),
               color='gray', linewidth=0.5, linestyle=':', alpha=0.6)

ax.set_title(
    'Serie Temporal Sintetica (semilla=42)\n'
    'Componentes: Tendencia lineal + Estacionalidad anual + Ciclo ~4 anos + Ruido gaussiano',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel('Fecha', fontsize=10)
ax.set_ylabel('Valor', fontsize=10)
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f'Valor inicial (2018-01-01): {serie.iloc[0]:.2f}')
print(f'Valor final  (2023-12-31): {serie.iloc[-1]:.2f}')
print(f'Incremento total         : {serie.iloc[-1] - serie.iloc[0]:.2f}')
print(f'Incremento teorico (0.05 x 2191 dias): {0.05 * 2190:.2f}')

### Interpretacion del grafico

**Tendencia**: claramente visible — la serie sube de ~50-60 en 2018 a ~150-160 en 2023.
El incremento total teorico es 0.05 × 2,190 dias = 109.5 unidades, consistente con lo observado.
La media movil (linea roja) confirma una tendencia **lineal creciente** sin aceleracion.

**Estacionalidad**: los picos y valles anuales son claramente visibles y regulares.
Se repiten con periodo de ~365 dias en cada ano del periodo analizado.

**Ciclo**: visible como una ondulacion de largo plazo sobre la tendencia. La media movil
no es perfectamente recta — hay una ligera curvatura cada ~4 anos que corresponde
al ciclo de amplitud 8 con periodo 1,461 dias.

**Ruido**: la irregularidad puntual alrededor de la estacionalidad — dispersion visible
pero contenida, consistente con sigma=3.5 teorico.

## TODO 2 — Descomposicion de la serie

### Que hace seasonal_decompose

`seasonal_decompose` de statsmodels aplica una descomposicion clasica por medias moviles:

1. **Estima la tendencia** con una media movil de `period` puntos (365 dias)
2. **Estima la estacionalidad** restando la tendencia y promediando por posicion dentro del periodo
3. **Calcula el residuo** como: `Y - Tendencia - Estacionalidad`

El parametro `period=365` le dice que el patron se repite cada 365 dias.  
Como usa una media movil de 365 puntos para la tendencia, **pierde 182 dias en cada extremo**
(la mitad del periodo) — por eso el residuo tiene NaN al inicio y al final.

### Parametros elegidos

- `model='additive'`: correcto porque la serie fue construida sumando componentes
- `period=365`: requerido por el enunciado, corresponde al periodo real de la estacionalidad

In [ ]:
resultado = seasonal_decompose(serie, model='additive', period=365)

fig = resultado.plot()
fig.set_size_inches(14, 10)

titulos = ['Serie original', 'Tendencia', 'Estacionalidad', 'Residuo']
for i, ax in enumerate(fig.axes):
    ax.set_title(titulos[i], fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

fig.suptitle(
    "Descomposicion Aditiva — seasonal_decompose(model='additive', period=365)",
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# Verificacion contra valores teoricos conocidos
tend = resultado.trend.dropna()
print('Verificacion contra componentes teoricos conocidos:')
print(f'  Tendencia recuperada — rango: {tend.min():.2f} a {tend.max():.2f}')
print(f'  Tendencia teorica    — rango: {50:.2f} a {0.05*2190+50:.2f}')
print(f'  Estacionalidad — amplitud recuperada: {resultado.seasonal.max():.2f}')
print(f'  Estacionalidad — amplitud teorica:    ~15 (componente dominante)')
print(f'  Residuo — media: {resultado.resid.mean():.4f} (teorico: 0)')
print(f'  Residuo — std  : {resultado.resid.std():.4f} (teorico: 3.5)')
print(f'  NaN en residuo : {resultado.resid.isna().sum()} observaciones')
print(f'  (182 en cada extremo por la media movil de 365 dias)')

### Interpretacion de la descomposicion

**Tendencia**: la descomposicion recupera correctamente la tendencia lineal creciente.
Rango de 64.14 a 155.25 — consistente con el teorico de 50 a ~159.5.
La pendiente estimada coincide con el 0.05/dia teorico.

**Estacionalidad**: amplitud recuperada ~14.46 unidades.  
El valor teorico es la amplitud de `15*sin(...) + 6*cos(...)`, que produce una amplitud
pico-a-pico de ~33 unidades (de -18 a +15 aproximadamente). El grafico muestra exactamente
ese patron: picos en verano (~+13) y valles en invierno (~-21).

**Residuo**: media=0.127 (muy cercana al ideal de 0) y std=3.22 (vs sigma teorico=3.5).
La diferencia entre 3.22 y 3.50 es esperable: la descomposicion absorbe algo del ruido
en los otros componentes al estimar por medias moviles.

**NaN en los extremos**: `seasonal_decompose` pierde 182 dias en cada extremo
(la mitad del periodo=365). De 2,191 observaciones, quedan 1,827 residuos validos
para el analisis. Esto es comportamiento esperado, no un error.

## TODO 3 — Analisis del residuo

### Criterios de ruido blanco gaussiano ideal

Un residuo de descomposicion ideal debe cumplir:

| Criterio | Test / Metrica | Valor ideal | Resultado obtenido |
|---|---|---|---|
| Media cero | media() | ≈ 0 | 0.127 ✓ |
| Distribucion normal | Jarque-Bera | p > 0.05 | p=0.577 ✓ |
| Estacionario | ADF | p < 0.05 | p=0.000 ✓ |
| Sin autocorrelacion | ACF/PACF | barras dentro del IC | Ver graficos ✓ |
| Simetrico | skewness | ≈ 0 | -0.051 ✓ |
| No leptocurtico | curtosis | ≈ 0 | -0.061 ✓ |

In [ ]:
residuo_limpio = resultado.resid.dropna()

media     = residuo_limpio.mean()
std       = residuo_limpio.std()
asimetria = residuo_limpio.skew()
curtosis  = residuo_limpio.kurtosis()

print(f'Estadisticos del residuo (n={len(residuo_limpio):,}):')
print(f'  Media     : {media:.6f}  (ideal: 0)')
print(f'  Std       : {std:.4f}  (sigma teorico: 3.5)')
print(f'  Asimetria : {asimetria:.4f}  (ideal: 0)')
print(f'  Curtosis  : {curtosis:.4f}  (ideal: 0 en exceso)')

# Jarque-Bera
jb_stat, jb_p = jarque_bera(residuo_limpio)
print(f'\nJarque-Bera: stat={jb_stat:.4f}, p={jb_p:.6f}')
print(f'  Conclusion: {"No se rechaza H0 -> normal" if jb_p > 0.05 else "Se rechaza H0 -> no normal"}')

# ADF
res_adf = adfuller(residuo_limpio, autolag='AIC')
print(f'\nADF: stat={res_adf[0]:.4f}, p={res_adf[1]:.6f}')
print(f'  Conclusion: {"Rechaza H0 -> estacionario" if res_adf[1] < 0.05 else "No rechaza H0"}')

### Interpretacion de los tests estadisticos

**Jarque-Bera p=0.577 >> 0.05**: no rechazamos H0.  
El residuo es compatible con una distribucion normal. Esto es el resultado esperado
porque la serie fue generada con ruido `N(0, 3.5)` — la descomposicion recupera
correctamente ese ruido gaussiano.

**ADF p=0.000 < 0.05**: rechazamos H0 con p-value practicamente cero.  
El estadistico ADF = -39.92 es extremadamente negativo (mas negativo = mas evidencia
de estacionariedad). El residuo ES estacionario — no tiene tendencia ni patron temporal.
Esto confirma que la descomposicion extrajo correctamente la tendencia y la estacionalidad.

**Skewness = -0.051 y curtosis = -0.061**: ambos practicamente cero.  
La distribucion es simetrica y de altura normal (no leptocurtica ni platicurtica).
Contrasta con el residuo del ejercicio 2 (skewness=-0.826, curtosis=7.67) que mostraba
desviaciones significativas de la normalidad.

In [ ]:
# ACF y PACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(residuo_limpio, lags=40, alpha=0.05, ax=axes[0],
         color='#E53935', vlines_kwargs={'colors': '#E53935'})
axes[0].set_title('ACF del Residuo\n(barras dentro del IC = sin autocorrelacion)',
                  fontsize=10, fontweight='bold')
axes[0].set_xlabel('Lag (dias)')
axes[0].grid(True, alpha=0.3)

plot_pacf(residuo_limpio, lags=40, alpha=0.05, ax=axes[1],
          color='#1976D2', vlines_kwargs={'colors': '#1976D2'}, method='ywm')
axes[1].set_title('PACF del Residuo\n(barras dentro del IC = sin autocorrelacion parcial)',
                  fontsize=10, fontweight='bold')
axes[1].set_xlabel('Lag (dias)')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Autocorrelacion del Residuo — Verificacion de Ruido Blanco',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Interpretacion ACF y PACF

**ACF**: todas las barras de lag=1 a lag=40 estan dentro del intervalo de confianza al 95%
(la banda azul sombreada). Solo la barra de lag=0 sobresale — eso es la autocorrelacion
de la serie consigo misma (siempre = 1), no es informativa.

**PACF**: mismo resultado — todas las barras dentro del IC.

**Conclusion**: el residuo no tiene autocorrelacion significativa en ningun lag de 1 a 40 dias.
Esto confirma que es **ruido blanco** — cada observacion es independiente de las anteriores.

Combinado con los tests anteriores: el residuo cumple todos los criterios de ruido
blanco gaussiano ideal. La descomposicion fue exitosa.

In [ ]:
# Histograma con curva normal
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(residuo_limpio, bins=50,
        color='#E53935', edgecolor='white', linewidth=0.5,
        alpha=0.75, density=True, label='Residuo observado')

x = np.linspace(residuo_limpio.min(), residuo_limpio.max(), 300)
ax.plot(x, norm.pdf(x, loc=media, scale=std),
        color='#43A047', linewidth=2.5,
        label=f'Normal ajustada N({media:.2f}, {std:.2f})')
ax.plot(x, norm.pdf(x, loc=0, scale=3.5),
        color='navy', linewidth=1.5, linestyle='--',
        label='Normal ideal N(0, 3.5) — sigma teorico')

ax.axvline(media, color='#E53935', linewidth=1.2, linestyle=':',
           label=f'Media residuo: {media:.4f}')
ax.axvline(0, color='black', linewidth=0.8, alpha=0.5, label='Media ideal = 0')

ax.set_xlabel('Residuo')
ax.set_ylabel('Densidad')
ax.set_title(
    f'Distribucion del Residuo vs Curva Normal Teorica\n'
    f'JB: stat={jb_stat:.2f}, p={jb_p:.4f} | ADF: p={res_adf[1]:.6f}',
    fontsize=10, fontweight='bold'
)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretacion del histograma

El histograma muestra tres curvas de referencia:

1. **Normal ajustada N(0.13, 3.22)** (linea verde solida): curva normal con la media
   y std reales del residuo. Se ajusta perfectamente al histograma — confirmando
   el resultado del Jarque-Bera.

2. **Normal ideal N(0, 3.5)** (linea azul discontinua): la distribucion teorica
   con la que se genero el ruido. Es ligeramente mas ancha porque sigma=3.5 > 3.22
   — la descomposicion absorbe una pequena parte del ruido en la tendencia estimada.

3. **Media del residuo** (linea roja punteada): 0.127, practicamente coincide con
   la media ideal de 0 (linea negra).

**Comparacion con el ejercicio 2**: el residuo de la regresion lineal (Ej.2) tenia
curtosis=7.67 y skewness=-0.83 — muy lejos de la normalidad. El residuo de esta
descomposicion tiene curtosis=-0.06 y skewness=-0.05 — practicamente normal perfecta.
La diferencia refleja que los datos sinteticos fueron generados exactamente para
producir ruido gaussiano, mientras que el dataset real tiene heterogeneidad no capturable
por un modelo lineal simple.

## Resumen para Respuestas.md

### Pregunta 4.1 — Tendencia

La serie presenta **tendencia lineal creciente**. Direccion: positiva.  
Magnitud: la tendencia crece de ~64 (inicio 2018) a ~155 (finales 2023),
un incremento de 91.1 unidades en 6 anos, equivalente a +0.05 unidades/dia (pendiente teorica).
La media movil de 365 dias y el componente de tendencia de `seasonal_decompose`
confirman el caracter estrictamente lineal sin aceleraciones.

### Pregunta 4.2 — Estacionalidad

Hay estacionalidad clara con **periodo de 365 dias** (anual).  
Amplitud: ~14.46 unidades (de -21 en invierno a +13 en verano).  
El patron se repite identico cada ano — consistente con la formula de generacion
que usa `sin(2π*t/365.25)`.

### Pregunta 4.3 — Ciclos de largo plazo

Se aprecian ciclos de largo plazo con periodo ~4 anos (1,461 dias) y amplitud 8 unidades.  
Se distinguen de la tendencia porque la tendencia es **monotonamente creciente**
mientras que el ciclo es **oscilatorio** (sube y baja). En la serie original se
ve como la velocidad de crecimiento no es constante — hay periodos donde la serie
sube mas rapido y otros donde se aplana ligeramente, correspondiendo al ciclo
superpuesto sobre la tendencia lineal.

### Pregunta 4.4 — Ruido ideal vs real

El residuo **se ajusta a ruido ideal** con los siguientes resultados:

| Criterio | Valor | Conclusion |
|---|---|---|
| Media | 0.127 | Cercana a 0 |
| Std | 3.22 | Cercana al sigma teorico 3.5 |
| Skewness | -0.051 | Practicamente simetrico |
| Curtosis | -0.061 | Practicamente mesocurtico |
| Jarque-Bera p | 0.577 | No rechaza normalidad (p > 0.05) |
| ADF p | 0.000 | Estacionario (p < 0.05) |
| ACF/PACF | todas barras en IC | Sin autocorrelacion significativa |

El residuo cumple todos los criterios de ruido blanco gaussiano ideal.
La pequena diferencia entre std=3.22 y sigma_teorico=3.50 se explica porque
`seasonal_decompose` absorbe una parte del ruido en la estimacion de la tendencia
mediante medias moviles.